# Lab 4: Database-Driven Feature Table


### 1. Explore the Schema
I am bringing in the database, finding the names of the tables in the database, and then looking at all of the columns in each table. The findings are including in my Lab-4 README.md file.

In [14]:
import sqlite3, pandas as pd
con = sqlite3.connect('cohort.db')
def q(sql): return pd.read_sql(sql, con)
q("SELECT name FROM sqlite_master WHERE type='table'")

,name
0,patients
1,samples
2,genes
3,mutations


In [15]:
patients = q("SELECT * FROM patients")
patients

,patient_id,sex,age_at_diagnosis,stage,vital_status,followup_months
0,TCGA-05-4244,male,70.0,IV,living,0.000000
1,TCGA-05-4249,male,67.0,IB,living,50.070684
2,TCGA-05-4250,female,79.0,IIIA,deceased,3.978039
3,TCGA-05-4382,male,68.0,IB,living,19.955946
4,TCGA-05-4384,male,66.0,IIIA,living,14.005326
...,...,...,...,...,...,...
561,TCGA-NJ-A55O,female,56.0,IIA,living,0.427393
562,TCGA-NJ-A55R,male,67.0,IA,living,19.824440
563,TCGA-NJ-A7XG,male,49.0,IIIA,living,20.284709
564,TCGA-O1-A52J,female,74.0,IA,deceased,59.111681


In [16]:
samples = q("SELECT * FROM samples")
samples

,sample_id,patient_id,tissue_source_site,tmb_nonsynonymous,aneuploidy_score,msi_mantis_score
0,TCGA-05-4244-01,TCGA-05-4244,Indivumed,6.400000,17.0,0.2886
1,TCGA-05-4249-01,TCGA-05-4249,Indivumed,10.000000,24.0,0.2807
2,TCGA-05-4250-01,TCGA-05-4250,Indivumed,10.500000,17.0,0.3059
3,TCGA-05-4382-01,TCGA-05-4382,Indivumed,51.733333,22.0,0.3193
4,TCGA-05-4384-01,TCGA-05-4384,Indivumed,3.966667,1.0,0.2850
...,...,...,...,...,...,...
561,TCGA-NJ-A55O-01,TCGA-NJ-A55O,Washington University - Rush University,4.133333,NaN,0.3210
562,TCGA-NJ-A55R-01,TCGA-NJ-A55R,Washington University - Rush University,7.700000,21.0,0.3064
563,TCGA-NJ-A7XG-01,TCGA-NJ-A7XG,Washington University - Rush University,0.900000,16.0,0.3235
564,TCGA-O1-A52J-01,TCGA-O1-A52J,Washington University - CALGB,5.100000,29.0,0.3220


In [17]:
mutations = q("SELECT * FROM mutations")
mutations

,mutation_id,sample_id,gene_symbol,variant_classification,variant_type,protein_change,chromosome,start_pos,end_pos,t_ref_count,t_alt_count
0,1,TCGA-05-4244-01,ARHGEF10L,Missense_Mutation,SNP,p.P964A,1,17990971,17990971,34,3
1,2,TCGA-05-4244-01,HTR6,Missense_Mutation,SNP,p.A263G,1,20005133,20005133,89,22
2,3,TCGA-05-4244-01,NR0B2,Missense_Mutation,SNP,p.P80S,1,27240194,27240194,24,10
3,4,TCGA-05-4244-01,AHDC1,Missense_Mutation,SNP,p.T1358I,1,27874554,27874554,83,30
4,5,TCGA-05-4244-01,SZT2,Silent,SNP,p.P675=,1,43888542,43888542,8,4
...,...,...,...,...,...,...,...,...,...,...,...
200784,200785,TCGA-S2-AA1A-01,GUCY2F,Splice_Site,SNP,p.X597_splice,X,108673535,108673535,200,19
200785,200786,TCGA-S2-AA1A-01,DDX26B,Missense_Mutation,SNP,p.D572N,X,134709092,134709092,131,11
200786,200787,TCGA-S2-AA1A-01,ARHGEF6,Missense_Mutation,SNP,p.D59Y,X,135861647,135861647,60,10
200787,200788,TCGA-S2-AA1A-01,SPANXN3,Silent,SNP,p.T13=,X,142605181,142605181,177,20


In [18]:
genes = q("SELECT * FROM genes")
genes

,gene_symbol,entrez_gene_id,chromosome
0,A1BG,1.0,19
1,A1CF,29974.0,10
2,A2M,2.0,12
3,A2ML1,144568.0,12
4,A3GALT2,127550.0,1
...,...,...,...
17751,ZXDC,79364.0,3
17752,ZYG11B,79699.0,1
17753,ZYX,7791.0,7
17754,ZZEF1,23140.0,17


### 2. Write SQL queries
Task One: For each patient and each of these five genes (TP53, KRAS, EGFR, STK11, KEAP1), count the protein-altering mutations (exclude Silent).

The row count of the task_one table is 707. If each patient (566) had a protein-altering mutation in each of the five genes of interest, then we would expect to see 2830 rows. However, many patients only have one gene mutation. Addiionally, not all patients have protein-altering mutations, and are not included in this table. Only 458 of the 566 patients have protein-altering mutations.

In [19]:
task_one = q('''SELECT p.patient_id, m.gene_symbol, COUNT(m.variant_classification) AS protein_altering_mutations
FROM patients p
LEFT JOIN samples s on s.patient_id=p.patient_id
LEFT JOIN mutations m on m.sample_id=s.sample_id
WHERE gene_symbol in ('TP53','KRAS','EGFR','STK11','KEAP1') AND (variant_classification != 'silent')
GROUP BY p.patient_id, m.gene_symbol''')

num_genes = task_one["gene_symbol"].nunique()
num_patients = patients["patient_id"].nunique()
print(task_one["patient_id"].nunique())
print(num_genes, num_patients)
print(num_genes * num_patients)
print(len(task_one))
task_one

458
5 566
2830
707


,patient_id,gene_symbol,protein_altering_mutations
0,TCGA-05-4244,KRAS,1
1,TCGA-05-4249,KRAS,1
2,TCGA-05-4250,KRAS,1
3,TCGA-05-4382,EGFR,2
4,TCGA-05-4382,TP53,1
...,...,...,...
702,TCGA-NJ-A55O,STK11,1
703,TCGA-NJ-A55R,KEAP1,1
704,TCGA-NJ-A55R,KRAS,1
705,TCGA-NJ-A55R,STK11,2


Task Two: Count patients per tissue source site per tumor stage.

In [20]:
task_two = q('''SELECT s.tissue_source_site, p.stage, COUNT(p.patient_id) AS patients
FROM samples s
LEFT JOIN patients p on p.patient_id=s.patient_id
GROUP BY s.tissue_source_site, p.stage''')

num_tissue_ss = task_two["tissue_source_site"].nunique()
num_stage = task_two["stage"].nunique()
print(num_tissue_ss, num_stage)
print(num_stage*num_tissue_ss)
print(len(task_two))
task_two.head(15)

34 9
306
142


,tissue_source_site,stage,patients
0,ABS - IUPUI,IA,5
1,ABS - IUPUI,IB,5
2,ABS - IUPUI,IIA,3
3,ABS - IUPUI,IIB,1
4,ABS - IUPUI,IIIA,2
5,ABS - Lahey Clinic,IA,3
6,ABS - Lahey Clinic,IIA,1
7,ABS - Lahey Clinic,IIB,1
8,Albert Einstein Medical Center,IA,1
9,Asterand,IA,6


### 3. Build a Feature Table

The original patient count is 566.

In [21]:
print(patients["patient_id"].nunique())

566


After my first join of the patients and samples tables, the patient count is still 566.

In [22]:
feature_table = q('''SELECT p.patient_id, p.sex, p.age_at_diagnosis, p.stage, s.tissue_source_site, s.tmb_nonsynonymous
FROM patients p
LEFT JOIN samples s on s.patient_id=p.patient_id''')

print(feature_table["patient_id"].nunique())
feature_table

566


,patient_id,sex,age_at_diagnosis,stage,tissue_source_site,tmb_nonsynonymous
0,TCGA-05-4244,male,70.0,IV,Indivumed,6.400000
1,TCGA-05-4249,male,67.0,IB,Indivumed,10.000000
2,TCGA-05-4250,female,79.0,IIIA,Indivumed,10.500000
3,TCGA-05-4382,male,68.0,IB,Indivumed,51.733333
4,TCGA-05-4384,male,66.0,IIIA,Indivumed,3.966667
...,...,...,...,...,...,...
561,TCGA-NJ-A55O,female,56.0,IIA,Washington University - Rush University,4.133333
562,TCGA-NJ-A55R,male,67.0,IA,Washington University - Rush University,7.700000
563,TCGA-NJ-A7XG,male,49.0,IIIA,Washington University - Rush University,0.900000
564,TCGA-O1-A52J,female,74.0,IA,Washington University - CALGB,5.100000


Pivoting the task_one table to have columns for each gene.

In [23]:
wide_task_one = task_one.pivot_table(index="patient_id", columns="gene_symbol" , values="protein_altering_mutations")
wide_task_one

gene_symbol,EGFR,KEAP1,KRAS,STK11,TP53
patient_id,,,,,
TCGA-05-4244,NaN,NaN,1.0,NaN,NaN
TCGA-05-4249,NaN,NaN,1.0,NaN,NaN
TCGA-05-4250,NaN,NaN,1.0,NaN,NaN
TCGA-05-4382,2.0,NaN,NaN,NaN,1.0
TCGA-05-4384,NaN,1.0,NaN,NaN,1.0
...,...,...,...,...,...
TCGA-NJ-A4YP,NaN,NaN,1.0,2.0,NaN
TCGA-NJ-A4YQ,NaN,1.0,NaN,NaN,NaN
TCGA-NJ-A55O,NaN,NaN,1.0,1.0,NaN


In [24]:
feature_table = feature_table.merge(wide_task_one.reset_index(), on="patient_id", how="left")
print(feature_table["patient_id"].nunique())
feature_table

566


,patient_id,sex,age_at_diagnosis,stage,tissue_source_site,tmb_nonsynonymous,EGFR,KEAP1,KRAS,STK11,TP53
0,TCGA-05-4244,male,70.0,IV,Indivumed,6.400000,NaN,NaN,1.0,NaN,NaN
1,TCGA-05-4249,male,67.0,IB,Indivumed,10.000000,NaN,NaN,1.0,NaN,NaN
2,TCGA-05-4250,female,79.0,IIIA,Indivumed,10.500000,NaN,NaN,1.0,NaN,NaN
3,TCGA-05-4382,male,68.0,IB,Indivumed,51.733333,2.0,NaN,NaN,NaN,1.0
4,TCGA-05-4384,male,66.0,IIIA,Indivumed,3.966667,NaN,1.0,NaN,NaN,1.0
...,...,...,...,...,...,...,...,...,...,...,...
561,TCGA-NJ-A55O,female,56.0,IIA,Washington University - Rush University,4.133333,NaN,NaN,1.0,1.0,NaN
562,TCGA-NJ-A55R,male,67.0,IA,Washington University - Rush University,7.700000,NaN,1.0,1.0,2.0,NaN
563,TCGA-NJ-A7XG,male,49.0,IIIA,Washington University - Rush University,0.900000,NaN,NaN,NaN,NaN,NaN
564,TCGA-O1-A52J,female,74.0,IA,Washington University - CALGB,5.100000,NaN,NaN,NaN,NaN,1.0


In [25]:
feature_table.to_csv("feature_table.tsv", sep="\t", index=False)

### 4. Handle Missing Data Explicitly
The final join between the feature table and the pivoted task one table maintains the 566 patients. I maintained the same number of patients by left joining all of my tables and keeping NaN in columns where patients did not have data.

### 5. Metadata

In [26]:
data_dict = {
    "column_name": feature_table.columns,
    "meaning": ['patient identifier', 
                'patient sex', 
                'age at diagnosis of Lung Adenocarcinoma',
                'tumor stage',
                'medical center where tissue data was collected',
                'tumor mutation burden',
                'EGFR gene',
                'KEAP1 gene',
                'KRAS gene',
                'STK11 gene',
                'TP53 gene'],
    "units": ['no unit',
              'no unit',
              'years',
              'no unit',
              'no unit',
              'mut/Mb',
              '# of people',
              '# of people',
              '# of people',
              '# of people',
              '# of people'],
    "source_table": ['patients',
                     'patients',
                     'patients',
                     'patients',
                     'samples',
                     'samples',
                     'genes',
                     'genes',
                     'genes',
                     'genes',
                     'genes']
}
data_dict = pd.DataFrame(data_dict)
data_dict

,column_name,meaning,units,source_table
0,patient_id,patient identifier,no unit,patients
1,sex,patient sex,no unit,patients
2,age_at_diagnosis,age at diagnosis of Lung Adenocarcinoma,years,patients
3,stage,tumor stage,no unit,patients
4,tissue_source_site,medical center where tissue data was collected,no unit,samples
5,tmb_nonsynonymous,tumor mutation burden,mut/Mb,samples
6,EGFR,EGFR gene,# of people,genes
7,KEAP1,KEAP1 gene,# of people,genes
8,KRAS,KRAS gene,# of people,genes
9,STK11,STK11 gene,# of people,genes


In [27]:
data_dict.to_csv("data_dictionary.tsv", sep="\t", index=False)